In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import statistics
from itertools import repeat
from functools import reduce
from operator import mul
from math import log

In [ ]:
# fetch dataset
congressional_voting_records = fetch_ucirepo(id=105)

# data (as pandas dataframes)
X = congressional_voting_records.data.features
y = congressional_voting_records.data.targets

df = pd.concat([X,y], axis=1)

In [ ]:
lambda_val = 1

In [ ]:
features = list(df.columns)[:-1]
feature_values = { feat: df[feat].unique() for feat in features}

In [ ]:
def get_classes_data(data):
  class_names = list(data["Class"].unique())
  return [
      {
          "class_title": _class,
          "class_table": data[data["Class"] == _class],
          "class_prob": (((data["Class"] == _class).sum() + lambda_val ) / (data.shape[0]+ lambda_val * len(class_names)))
      } for _class in class_names
  ]

In [ ]:
def get_probs(data):
  classes = get_classes_data(data)
  return {
      feat:
       {
           feat_value:
            {
                _class.get("class_title"):
                      (((_class.get("class_table")[feat] ==  feat_value).sum()+ lambda_val) / (_class.get("class_table").shape[0] + lambda_val * len(data[feat].unique())))
                 for _class in classes
            } for feat_value in feature_values[feat]
        } for feat in features}

In [ ]:
def estimate(row,classes,probs):
  estimates = {
       _class.get("class_title"):
           sum([log(probs[feat][list(row[feat])[0]][_class.get("class_title")],2) for feat in features] + [log(_class.get("class_prob"),2)])
        for  _class in classes
  }
  final_result = max(estimates, key = estimates.get)
  return final_result

In [ ]:
def isCorrectEstimate(row,classes,probs):
  estimate_result = estimate(row,classes,probs)
  actual_result = row["Class"]
  return estimate_result == actual_result

In [ ]:
def determine_accuracy(train_data, test_data):
  classes = get_classes_data(train_data)
  probs = get_probs(train_data)
  accuracy = sum([isCorrectEstimate(row,classes,probs) for index,row in test_data.iterrows()]) / test_data.shape[0]
  return accuracy

In [ ]:
def get_train_test_sets(data):
  classes = get_classes_data(data);
  train_set = []
  test_set = []
  for _class in classes:
    entries = _class.get("class_table").shape[0]
    indices = list(range(entries))
    random.shuffle(indices)
    k = int(entries * 0.9)
    train_set.extend(data.iloc[indices[:k]].to_dict('records'))
    test_set.extend(data.iloc[indices[k:]].to_dict('records'))

  return pd.DataFrame(train_set), pd.DataFrame(test_set)

In [ ]:
def get_stratified_k_folds(data, k):
  classes = get_classes_data(data);
  folds = [ [] for _ in range(k)]
  for _class in classes:
    entries = _class.get("class_table").shape[0]
    entries_per_fold = int(entries / k)
    indices = list(range(entries))
    random.shuffle(indices)
    current_fold = 0
    current_items_count = 0
    for i in range(entries):
      folds[current_fold].append(data.iloc[indices[i]].reset_index(drop=True))
      current_items_count+=1
      if(current_items_count == entries_per_fold):
        if(current_fold == (k - 1)):
          current_fold = -1
          entries_per_fold = 1
        current_items_count = 0
        current_fold+=1
  return [pd.DataFrame(fold) for fold in folds]

In [ ]:
def perform_k_fold_cross_validation(data, k) :
  print(f"------ Performing {k}-Fold Cross-Validation ------")
  folds = get_stratified_k_folds(data, k)
  accuracies = []
  for index, fold in enumerate(folds):
    train_data = pd.concat(folds[:index]+folds[index+1:])
    train_data = train_data.set_axis(features + ["Class"], axis=1)

    test_data = folds[index]
    test_data = test_data.set_axis(features + ["Class"], axis=1)

    accuracy = determine_accuracy(train_data, test_data)
    accuracies.append(accuracy)
    print(f"[FOLD {index}] Accuracy: {accuracy:.2%}")

  print(f"Average acuracy: {statistics.mean(accuracies):.2%}")
  print(f"Standard deviation: {statistics.stdev(accuracies):.2%}")
  print(f"------ Validation completed ------")


In [ ]:
def get_complete_data(data, use_third_option = False):
  result = data.copy(deep=True)

  if use_third_option:
    # use missing value as third option
    result = result.fillna('k')
  else:
    # replace with the most frequent value in the column
    for column in result.columns:
      mode = result[column].mode().iloc[0]
      result = result.fillna({column:mode})

  global feature_values
  feature_values = { feat: result[feat].unique() for feat in features}

  return result


In [ ]:
random.seed()
lambda_val = 0.1
data = get_complete_data(df, use_third_option=True)

train_set, test_set = get_train_test_sets(data)
print (f"Train set accuracy: {determine_accuracy(train_set, train_set):.2%}")
perform_k_fold_cross_validation(train_set,10)
print (f"Test set accuracy: {determine_accuracy(train_set, test_set):.2%}")

Train set accuracy: 89.77%
------ Performing 10-Fold Cross-Validation ------
[FOLD 0] Accuracy: 92.50%
[FOLD 1] Accuracy: 95.00%
[FOLD 2] Accuracy: 85.00%
[FOLD 3] Accuracy: 97.44%
[FOLD 4] Accuracy: 89.74%
[FOLD 5] Accuracy: 94.87%
[FOLD 6] Accuracy: 89.74%
[FOLD 7] Accuracy: 94.87%
[FOLD 8] Accuracy: 84.21%
[FOLD 9] Accuracy: 84.21%
Average acuracy: 90.76%
Standard deviation: 4.95%
------ Validation completed ------
Test set accuracy: 90.91%
